In [1]:
%%capture

import altair as alt
import gcsfs
import pandas as pd

#from calitp_data_analysis import calitp_color_palette as cp
from IPython.display import HTML, Markdown, display
#from update_vars import GCS_FILE_PATH, MONTH, PUBLIC_FILENAME, YEAR
#from _01_ntd_ridership_utils import sum_by_group
from gtfs_curator_utils import magics

alt.data_transformers.enable("vegafusion")

WIDTH = 300
HEIGHT = 150

In [2]:
# parameters cell for local
rtpa = "Metropolitan Transportation Commission"

In [3]:
%%capture_parameters
rtpa

{"rtpa": "Metropolitan Transportation Commission"}


# {rtpa}
Annual Ridership Trends

Download data from our **[public folder](https://console.cloud.google.com/storage/browser/calitp-publish-data-analysis)** by navigating to `ntd_annual_ridership` and selecting a file.

Transit operators/agencies that submit annual reports to NTD are included in this report. Reporters that were previously active reporters, but are currently not, may appear. This may result in Reporters showing zero or partial ridership data in the report.

If a Reporter, type of service, mode, or any combination of, is not a annual reporter or has not reported data since 2018, they will not appear in the report.

Examples:

* **Reporter A** is an annual reporter from 2019-2022, then became inactive and did not report for 2023. Reporter A's ridership data will be displayed for 2019-2022 only.
* **Reporter B** is an annual from 2000-2017, then became inactive and did not report for 2018. Reporter B will be named in the report, but will not display ridership data.
* **Reporter C** was an inactive reporter form 2015-2020, then became an active full reporter for 2021. Reporter C's ridership data will be displayed for 2021-present.


# need to set PUBLIC_FILENAME in update_vars
URL = "https://console.cloud.google.com/storage/" "browser/calitp-publish-data-analysis"

display(
    HTML(
        f"""
        <a href={URL}>
        Download the latest month of data: {PUBLIC_FILENAME}</a>
        """
    )
)

* [annual ridership query](https://github.com/tiffanychu90/curator/blob/use-new-ntd-tables/ntd/ntd_utils.py#L293)
   * includes `agency_status`, what is this? 

In [97]:
# read in data
GCS_FILE_PATH = "gs://calitp-analytics-data/data-analyses/ntd_explore/"

# annual uses rtpa_name or rtpa_name_split? 
# monthly uses ?
crosswalk = pd.read_parquet(
    f"{GCS_FILE_PATH}crosswalk.parquet", 
    filesystem=gcsfs.GCSFileSystem(),
    filters = [[("rtpa_name", "==", rtpa)]]
).rename(columns = {"ntd_id_2022": "ntd_id"}).drop_duplicates()

full_df = pd.read_parquet(f"{GCS_FILE_PATH}annual.parquet",filesystem=gcsfs.GCSFileSystem())

df = pd.read_parquet(
    f"{GCS_FILE_PATH}annual.parquet",
    filesystem=gcsfs.GCSFileSystem(),
).merge(
    crosswalk,
    on = "ntd_id",
    how = "inner"
)

In [92]:
df.columns

Index(['key', 'ntd_id', 'mode', 'year', 'type_of_service',
       'unlinked_passenger_trips', 'vehicle_revenue_hours',
       'vehicle_revenue_miles', 'vehicles_operated_in_maxiumum_service',
       'passenger_miles_traveled', 'direction_route_miles',
       'operating_expenses_vehicle_operations',
       'operating_expenses_vehicle_maintenance',
       'operating_expenses_nonvehicle_maintenance',
       'operating_expenses_general_administration', 'operating_expenses_total',
       'fare_revenue', 'opex_per_vrh', 'opex_per_vrm', 'opex_per_upt',
       'upt_per_vrh', 'upt_per_vrm', 'farebox_recovery_ratio', 'agency_status',
       'census_year', 'last_report_year', 'mode_status', 'reporter_type',
       'reporting_module', 'uace_code', 'uza_area_sq_miles',
       'primary_uza_name', 'uza_population', 'source_agency', 'source_city',
       'source_state', 'upt_prior_year', 'upt_change_1yr',
       'upt_pct_change_1yr', 'organization_name', 'county_name', 'rtpa_name',
       'rtpa_name_s

In [93]:
one_agency = "City and County of San Francisco (SFMTA) - Transit Division"
test_df = df[df.source_agency==one_agency].reset_index(drop=True)

In [94]:
import B3_ntd_utils as ntd_utils

In [95]:
df.source_agency.nunique()
# existing report has 23, where are these 3?

20

In [103]:
crosswalk[(full_df.source_agency.str.contains("Water Emergency")) |
    (full_df.source_agency.str.contains("County of Sonoma")) |
    (full_df.source_agency.str.contains("MTC"))]

(22, 5)

In [105]:
crosswalk.ntd_id.value_counts()

ntd_id
90144    1
90161    1
90017    1
90213    1
90232    1
90155    1
90092    1
91014    1
90013    1
90009    1
90134    1
90003    1
90016    1
90015    1
90088    1
90234    1
90078    1
90162    1
90159    1
90150    1
90014    1
90299    1
Name: count, dtype: int64

In [104]:
full_df[(full_df.source_agency.str.contains("Water Emergency")) |
    (full_df.source_agency.str.contains("County of Sonoma")) |
    (full_df.source_agency.str.contains("MTC"))].ntd_id.value_counts()

ntd_id
90089    21
90225    14
90094     7
Name: count, dtype: int64

In [89]:
# looks like SFMTA is the one that's incorrect
# should be 1_083_113_299
# getting 3_249_339_897
df.pipe(proportion_of_upt_by_agency)
# missing
# San Francisco Bay Area Water Emergency Transpo.
# County of Sonoma (SCT) - Department of Public 	
# Metropolitan Transportation Commission (MTC) -... 	

,source_agency,total_upt,pct_of_total_upt
2,City and County of San Francisco (SFMTA) - Tra...,1083113299,46.93
13,San Francisco Bay Area Rapid Transit District ...,510023863,22.10
0,Alameda-Contra Costa Transit District,278039351,12.05
15,Santa Clara Valley Transportation Authority (VTA),184029668,7.97
12,Peninsula Corridor Joint Powers Board (PCJPB),71071318,3.08
14,San Mateo County Transit District (SMCTD),63179294,2.74
8,"Golden Gate Bridge, Highway and Transportation...",23248508,1.01
1,Central Contra Costa Transit Authority (CCCTA),18271716,0.79
10,Marin County Transit District (MCTD),18026007,0.78
18,The Eastern Contra Costa Transit Authority,10387647,0.45


In [87]:
df.pipe(proportion_of_upt_by_agency).total_upt.sum()
# 2,331,534,369

2308006739

In [88]:
# this is total upt since 2018, which is a parameter in the query
# might need to set this in update_vars, otherwise if it updates, we don't know and caption is wrong
# These counts are totally over counting, more than double, look into why
def proportion_of_upt_by_agency(df: pd.DataFrame):
    initial_agg = (
        df
        .groupby("source_agency")
        .agg(
            total_upt=("unlinked_passenger_trips", "sum")
        ).reset_index()
        .astype({"total_upt": "int64"})
        .sort_values(by="total_upt", ascending=False)
    )
     # % total columns
    initial_agg["pct_of_total_upt"] = ((
        initial_agg["total_upt"] / initial_agg["total_upt"].sum()
    ) * 100).round(decimals=2)

    return initial_agg
    
